# Zed – trénink modelu Zeds World

QLoRA fine-tune **Qwen3.5 4B** na lore, roleplay a GDScriptu z repozitáře.

**Než spustíš:** nahoře *Runtime → Change runtime type → T4 GPU*. Bez GPU to nepojede.

Celé to trvá kolem hodiny a půl. Buňky pouštěj popořadě.

## 1. Instalace

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps --upgrade "trl>=0.15.0" peft accelerate bitsandbytes

import torch
assert torch.cuda.is_available(), "Není GPU – Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Dataset

Nahraj `dataset.jsonl` (vyrobí ho `build_dataset.py`) přes panel souborů vlevo,
nebo si ho stáhni z GitHubu.

In [ ]:
# Varianta A – z GitHubu (uprav branch, pokud je jiný):
# !wget -q -O dataset.jsonl https://raw.githubusercontent.com/jackhacker2021/project/ccr-4ad76bcf-mf7gfj/zedsworld_ai/data/dataset.jsonl

# Varianta B – nahraj soubor ručně vlevo do /content

from datasets import load_dataset

full = load_dataset("json", data_files="dataset.jsonl", split="train")

# Desetina stranou na ověřování. Tohle je to, co minule chybělo:
# bez něj se nepozná, že se model učí data nazpaměť místo aby uměl psát.
split = full.train_test_split(test_size=0.1, seed=1337)
ds_train, ds_eval = split["train"], split["test"]
print(f"trénink {len(ds_train)}, ověření {len(ds_eval)}")

## 3. Model

Qwen3.5 4B ve 4 bitech. Pro telefon s 8 GB RAM je to správná velikost –
v Q4 má hotový model kolem 2,5 GB. Větší modely (9B a výš) se do telefonu nevejdou.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3.5-4B",
    max_seq_length = MAX_SEQ,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 64,
    lora_dropout = 0.05,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 1337,
)

## 4. Formát promptu

Musí sedět s tím, co staví aplikace v `ChatEngine.kt`, jinak se fine-tune nechytne.

In [ ]:
def to_text(batch):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in batch["messages"]
    ]}

ds_train = ds_train.map(to_text, batched=True, remove_columns=ds_train.column_names)
ds_eval  = ds_eval.map(to_text,  batched=True, remove_columns=ds_eval.column_names)
print(ds_train[0]["text"][:400])

## 5. Trénink

**Sleduj `eval_loss`, ne `loss`.** Trénovací loss klesá vždycky – i když se model
jen učí data nazpaměť. Jakmile `eval_loss` přestane klesat a začne růst, model se
přeučuje a trénink se sám zastaví (`EarlyStoppingCallback`).

Přesně tohle shodilo minulý pokus na malém modelu: trénovací loss byl 0,11,
přesnost 98 %, a model mluvil nesmysly.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_train,
    eval_dataset = ds_eval,
    args = SFTConfig(
        output_dir = "out",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_ratio = 0.05,
        lr_scheduler_type = "cosine",
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 20,
        save_total_limit = 2,
        load_best_model_at_end = True,      # vezme nejlepší, ne poslední
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        max_seq_length = MAX_SEQ,
        dataset_text_field = "text",
        packing = True,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        report_to = "none",
        seed = 1337,
    ),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 6. Zkouška – mluví jako Zed?

Tohle je skutečný test, ne čísla z tréninku. Když odpovědi nedávají smysl,
nemá cenu pokračovat na export.

In [ ]:
FastLanguageModel.for_inference(model)

import json
SYSTEM = json.loads(open("dataset.jsonl").readline())["messages"][0]["content"]

OTAZKY = [
    "Kdo je Zéd Zédovský? Odpověz stručně.",
    "Co znamená slovo ALIVER a co dělá?",
    "Je Zeds World skutečný svět, nebo vymyšlený?",
    "Kdo jsou Monder a Mazlamondr?",
    "Napiš GDScript pro Godot 4.3: CharacterBody2D, který se hýbe šipkami.",
]

for q in OTAZKY:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    out = model.generate(ids, max_new_tokens=220, temperature=0.7, do_sample=True)
    print("="*70)
    print(q)
    print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

## 7. Export do GGUF

Sloučí adaptér do modelu a zkvantizuje na Q4_K_M – kolem 2,5 GB.
Trvá to asi deset minut.

In [ ]:
model.save_pretrained_gguf("zed-4b", tokenizer, quantization_method="q4_k_m")
!ls -lh zed-4b/*.gguf

## 8. Stažení

Přes prohlížeč se 2,5 GB stahuje nespolehlivě – přes Drive to je jistější.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp zed-4b/*.gguf /content/drive/MyDrive/
print("hotovo – soubor je na Disku")

---

## Dál

Soubor přejmenuj na `zed-4b-q4_k_m.gguf` a buď:

- nahraj někam, odkud ho aplikace stáhne (uprav `ModelRepository.DEFAULT_URL`), nebo
- dej ho do `android/app/src/main/assets/` a aplikaci přelož znovu – bude v APK.